In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mahasaad12/verilog-example/counter.v


# Mistral (Large Language Model)

In [2]:
!pip install transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 107.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
from huggingface_hub import login
login()

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# Read verilog file 

In [6]:
with open("/kaggle/input/datasets/mahasaad12/verilog-example/counter.v", "r") as f:
    rtl = f.read()

print(rtl)

module counter(
    input clk,
    input rst,
    output reg [3:0] count
);

always @(posedge clk)
begin
    if(rst)
        count<=0;
    else
        count<=count+1;
end



# prompt

In [13]:
prompt = f"""
You are an ASIC RTL documentation assistant.

Your task is to analyze the Verilog module below.

Rules:
1. Do not invent signals.
2. Use only information found in the RTL.
3. If information is unavailable, write "Not specified".
4. Output only the requested documentation.

Format:

Module:

Inputs:

Outputs:

Internal Registers:

Internal Wires:

Always Blocks:

Description:

RTL:

{rtl}
"""

# Generate Text 

In [15]:
def generate_text(prompt, max_new_tokens=300):

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.2,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [16]:
summary = generate_text(prompt)

print(summary)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2479: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(



You are an ASIC RTL documentation assistant.

Your task is to analyze the Verilog module below.

Rules:
1. Do not invent signals.
2. Use only information found in the RTL.
3. If information is unavailable, write "Not specified".
4. Output only the requested documentation.

Format:

Module:

Inputs:

Outputs:

Internal Registers:

Internal Wires:

Always Blocks:

Description:

RTL:

module counter(
    input clk,
    input rst,
    output reg [3:0] count
);

always @(posedge clk)
begin
    if(rst)
        count<=0;
    else
        count<=count+1;
end

endmodule

---

Module: counter

Inputs:
- clk: Not specified
- rst: Not specified

Outputs:
- count: 4-bit unsigned integer

Internal Registers:
- count: 4-bit unsigned integer

Internal Wires:
- None

Always Blocks:
- Always block 1: Updates count on positive edge of clk if rst is not asserted. If rst is asserted, count is reset to 0.

Description:
The counter module increments its 4-bit count output on every positive edge of the clk i